# SilverWork_incremental_industrial_v2

**- Incremental Silver processing using only new Bronze rows.**

### Step 1

**This cell imports Spark,Window and Delta helpers,switches to the right catalaog, makes sure the Silver schema exists,creates a `silver_run_id` for the current run.**

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog venk_novacart")
spark.sql("create schema if not exists silver_schema")

silver_run_id = str(uuid.uuid4())
print("current Silver Run ID: ", silver_run_id)



# **Step 2 -- Silver control table**
This table stores the latest silver processing state for each entity

it helps us track:
- the latest Bronze run already processed by silver
- the latest Bronze ingestion timestamp already processed
- how many rows were merged in the latest silver run

In [0]:
spark.sql("""
        create table if not exists venk_novacart.silver_schema.processing_control (
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_bronze_ingested_at timestamp,
            rows_merged bigint,
            run_status string,
            silver_run_id string,
            updated_at timestamp    
        )
        using delta
""")

# **Step 3 -- helper functions**

This cell contains reusable logic for silver:
- `upsert_to_silver()` merges cleaned /transformed rows into the silver target table
- `get_last_processed_bronze_ingested_at()` reads silver watermark
- `upsert_silver_control()` updates the silver control table
- `get_incremental_bronze()` reads only new Bronze rows that silver has not processed yet

In [0]:
def upsert_to_silver(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
        .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    else:
        df_source.write.saveAsTable(target_table)
        

In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
    ctrl = (spark.table("venk_novacart.silver_schema.processing_control")
    .filter(
        (F.col("layer") == "silver") &
        (F.col("entity_name") == entity_name) &
        (F.col("run_status") == "SUCCESS")
    )
    .orderby(F.col("updated_at").desc())
    .limit(1))
    
    rows = ctrl.collect()
    if len(rows) == 0:
        return None
    else:
        return rows[0]["last_processed_bronze_ingested_at"]

In [0]:
def upsert_silver_table(entity_name,last_processed_bronze_run_id,last_processed_bronze_ingested_at,rows_merged):
    ctrl_df = spark.createDataFrame(
        [(
            "silver",
            entity_name,
            last_processed_bronze_run_id,
            last_processed_bronze_ingested_at,
            int(rows_merged),
            "SUCCESS",
            silver_run_id,
            datetime.utcnow()
        )],
        schema="""
                layer string,
                entity_name string,
                last_processed_bronze_run_id string,
                last_processed_bronze_ingested_at timestamp,
                rows_merged bigint,
                run_status string,
                silver_run_id string,
                updated_at timestamp"""
    ) 
    dt = DeltaTable.forName(spark, "venk_novacart.silver_schema.processing_control")
    (dt.alias("t")
         .merge(ctrl_df.alias("s"), "t.entity_name = s.entity_name")
         .whenMatchedUpdate(set = {last_processed_bronze_run_id:"s.last_processed_bronze_run_id",last_processed_bronze_ingested_at:"s.last_processed_bronze_ingested_at","rows_merged":"s.rows_merged","run_status":"s.run_status","silver_run_id":"s.silver_run_id","updated_at":"s.updated_at"})
         .whenNotMatchedInsertAll()
         .execute())

    
        

In [0]:
def get_incremental_data(bronze_table,entity_name):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.table(bronze_table)
    if last_ingested_at is None:
        return bronze_df,last_ingested_at
    else:
        return bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at)),last_ingested_at